# Limpieza de Datos: Perfil de Riesgo en una App de Inversion

**Curso:** Analitica de Datos / Fundamentos de Ciencia de Datos
**Tema:** Preprocesamiento y Limpieza de Datos
**Paqueteria principal:** `pandas`, `numpy`

## Contexto de negocio

**FinNova** es una fintech que opera una app de inversion para clientes retail. El equipo de
producto quiere entender que caracteristicas de los clientes (edad, ingreso, monto invertido,
antiguedad en la app) se asocian con cada **perfil de riesgo** (Conservador, Moderado,
Agresivo), para poder personalizar las recomendaciones de portafolio dentro de la app.

Te entregan un extracto de la base de clientes tal como sale del sistema. El problema: los
nombres de columna son inconsistentes, hay valores faltantes, registros duplicados y algunos
valores fuera de rango logico. Tu tarea es **limpiar la base paso a paso** y, al final,
construir una tabla cruzada que compare perfil de riesgo con rango de edad.

### Diccionario de datos (nombres originales, tal como llegan en el csv)

| Columna original | Descripcion |
|---|---|
| `ID Cliente` | Identificador del cliente |
| `Edad ` | Edad del cliente en anios (nota el espacio al final del nombre) |
| `Ingreso Mensual (MXN)` | Ingreso mensual reportado, en pesos mexicanos |
| `Antigüedad App (meses)` | Meses desde que el cliente se registro en la app |
| `Monto Invertido (MXN)` | Monto total invertido actualmente, en pesos mexicanos |
| `Perfil de Riesgo` | Conservador, Moderado o Agresivo |
| `Num. Transacciones` | Numero de transacciones realizadas en la app |
| `Ciudad` | Ciudad del cliente |

### Dataset

Los datos estan en `data/app_inversion_clientes_raw.csv`, en una carpeta relativa a este notebook.


### Objetivos de aprendizaje

Al terminar este notebook deberias poder:

1. Describir las caracteristicas basicas de un dataset (forma, tipos de dato).
2. Hacer una exploracion (EDA) numerica minima con `pandas`.
3. Homologar nombres de columna a un formato limpio y consistente.
4. Detectar y eliminar registros duplicados.
5. Detectar, cuantificar y tratar valores faltantes.
6. Detectar valores atipicos (outliers) con un criterio logico y con el criterio IQR.
7. Construir una tabla cruzada (`pd.crosstab`) para responder una pregunta de negocio.


### Como trabajar este notebook

- Las **15 preguntas** de este notebook tienen una **respuesta numerica exacta** (un entero o
  un decimal). No hay preguntas de interpretacion abierta.
- Cada pregunta tiene una celda de codigo vacia justo debajo. Escribe ahi tu solucion y
  termina la celda con un `print()` claro que muestre el numero pedido, por ejemplo:
  `print("Respuesta:", valor)`.
- Trabaja las secciones **en orden**: varias preguntas sobrescriben `df` y usan el resultado
  de la limpieza aplicada en la pregunta anterior.
- Esta practica esta disenada para resolverse en aproximadamente **30 minutos**.


## 1. Carga y forma de la base

### Pregunta 1

Carga `data/app_inversion_clientes_raw.csv` en un DataFrame llamado `df` (usa los nombres de columna tal como vienen en el archivo, no los cambies todavia). ¿Cuantas filas tiene el dataset?

In [2]:
# Tu codigo aqui
import pandas as pd
df = pd.read_csv("app_inversion_clientes_raw.csv")


In [3]:
df.shape[0]

315

### Pregunta 2

¿Cuantas columnas tiene el dataset?

In [4]:
# Tu codigo aqui
df.shape[1]

8

## 2. Caracteristicas de la base y exploracion (EDA)

### Pregunta 3

Usa `df.dtypes`. ¿Cuantas columnas tienen un tipo de dato numerico (`int64` o `float64`)?

In [5]:
# Tu codigo aqui
df.dtypes

ID Cliente                  int64
Edad                      float64
Ingreso Mensual (MXN)     float64
Antigüedad App (meses)      int64
Monto Invertido (MXN)     float64
Perfil de Riesgo              str
Num. Transacciones          int64
Ciudad                        str
dtype: object

In [6]:
df.select_dtypes(include=["int64","float64"]).shape[1]

6

### Pregunta 4

¿Cual es el promedio de la columna `"Ingreso Mensual (MXN)"`, redondeado a 2 decimales? (usa `.mean()`, que ignora los valores faltantes automaticamente)

In [7]:
# Tu codigo aqui
# Redondear a 2 decimales la columna "Ingreso Mensual (MXN)
promedio=df["Ingreso Mensual (MXN)"].mean()
print(promedio)


25096.4602739726


In [8]:
round(promedio,2)


np.float64(25096.46)

### Pregunta 5

¿Cual es la mediana de la columna `"Monto Invertido (MXN)"`, redondeada a 2 decimales?

In [9]:
# Tu codigo aqui
mediana = df["Monto Invertido (MXN)"].median()
round(mediana, 2)

np.float64(9870.76)

## 3. Homologacion de nombres de columnas

### Pregunta 6

Los nombres de columna originales tienen espacios, mayusculas, acentos y parentesis inconsistentes (por ejemplo `"Edad "` con un espacio al final, o `"Ingreso Mensual (MXN)"`). Renombra **todas** las columnas a formato `snake_case` limpio: `id_cliente`, `edad`, `ingreso_mensual`, `antiguedad_meses`, `monto_invertido`, `perfil_riesgo`, `num_transacciones`, `ciudad`. Sobrescribe `df` con el resultado. ¿Cuantas de las 8 columnas originales tenian un nombre **distinto** al nombre final homologado (es decir, cuantas tuviste que corregir)?

In [10]:
# Tu codigo aqui
list(df.columns)

['ID Cliente',
 'Edad ',
 'Ingreso Mensual (MXN)',
 'Antigüedad App (meses)',
 'Monto Invertido (MXN)',
 'Perfil de Riesgo',
 'Num. Transacciones',
 'Ciudad']

In [21]:
df.columns = [
    "id_cliente",
    "edad",
    "ingreso_mensual",
    "antiguedad_meses",
    "monto_invertido",
    "perfil_riesgo",
    "num_transacciones",
    "ciudad"
]

df.columns

Index(['id_cliente', 'edad', 'ingreso_mensual', 'antiguedad_meses',
       'monto_invertido', 'perfil_riesgo', 'num_transacciones', 'ciudad'],
      dtype='str')

## 4. Duplicados

### Pregunta 7

¿Cuantas filas duplicadas exactas (todas las columnas iguales) tiene el DataFrame?

In [22]:
# Tu codigo aqui

df.duplicated().sum()


np.int64(15)

In [23]:
df[df.duplicated()]

,id_cliente,edad,ingreso_mensual,antiguedad_meses,monto_invertido,perfil_riesgo,num_transacciones,ciudad
75,1178,70.0,25085.44,34,5058.34,Conservador,11,Guadalajara
84,1281,56.0,19765.20,14,10311.87,Moderado,15,Chihuahua
117,1082,22.0,34329.75,36,NaN,Conservador,4,Monterrey
118,1205,19.0,10927.38,35,6477.05,Agresivo,9,Guadalajara
139,1238,NaN,7943.87,18,5411.06,Moderado,11,Ciudad Juarez
167,1274,19.0,26320.94,43,3896.31,Conservador,10,Ciudad Juarez
203,1038,200.0,23859.68,7,10435.37,Moderado,11,Chihuahua
211,1151,70.0,NaN,24,9980.82,Moderado,10,Monterrey
214,1067,28.0,22701.58,1,14005.01,Agresivo,18,Chihuahua
245,1109,51.0,12810.70,44,2891.99,Conservador,11,Monterrey


### Pregunta 8

Elimina las filas duplicadas exactas con `drop_duplicates()` y sobrescribe `df` con el resultado (usa `reset_index(drop=True)`). ¿Cuantas filas quedan en el DataFrame despues de eliminarlas?

In [24]:
# Tu codigo aqui
df = df.drop_duplicates().reset_index(drop=True)

In [25]:
df.drop_duplicates()

,id_cliente,edad,ingreso_mensual,antiguedad_meses,monto_invertido,perfil_riesgo,num_transacciones,ciudad
0,1051,38.0,NaN,26,3718.49,Conservador,7,Chihuahua
1,1299,68.0,28682.80,32,10307.92,Moderado,9,Chihuahua
2,1252,26.0,36902.82,24,26134.54,Agresivo,12,Guadalajara
3,1232,37.0,32278.12,24,14014.61,Moderado,11,CDMX
4,1016,34.0,42062.61,30,15694.91,Moderado,5,Chihuahua
...,...,...,...,...,...,...,...,...
295,1202,50.0,8182.86,19,500.00,Moderado,11,Chihuahua
296,1169,67.0,28230.71,24,8349.99,Conservador,9,CDMX
297,1186,56.0,30005.14,15,8095.01,Conservador,17,Chihuahua
298,1036,NaN,21110.26,30,8073.60,Moderado,21,Ciudad Juarez


In [26]:
df.shape[0]

300

In [27]:
len(df)

300

## 5. Valores faltantes

### Pregunta 9

Sobre el DataFrame sin duplicados, ¿cuantos valores faltantes (NaN) hay **en total**, sumando todas las columnas?

In [28]:
# Tu codigo aqui
df["edad"].unique()

array([ 38.,  68.,  26.,  37.,  34.,  43.,  42.,  27.,  70.,  33.,  39.,
        47.,  46.,  20.,  nan,  69.,  52.,  74.,  53.,  21.,  73.,  55.,
        71.,  28.,  66.,  18.,  45.,  19.,  48.,  25.,  40.,  22.,  51.,
        56.,  67.,  57.,  61.,  63.,  72.,  32.,  54.,  29.,  64.,  49.,
        50.,  41.,  24.,  44.,  65.,  35.,  36., 200.,  30.,  58.,  60.,
        62.,  31.,  23., 150.,  -1.,  59.,  -5.])

In [30]:
df["edad"].isna().sum()

np.int64(18)

### Pregunta 10

¿Cuantos valores faltantes tiene especificamente la columna `edad`?

In [31]:
# Tu codigo aqui
mediana = df["edad"].median()
mediana


np.float64(43.5)

In [32]:
df["edad"] = df["edad"].fillna(mediana)

In [33]:
mediana = df["edad"].median()
mediana


np.float64(43.5)

### Pregunta 11

Calcula la mediana de la columna `edad` (ignorando los NaN) y usala para rellenar (`fillna`) los valores faltantes de esa columna. Sobrescribe la columna `edad` en `df`. ¿Cual es el valor exacto (con decimales si aplica) que usaste para la imputacion?

In [16]:
# Tu codigo aqui


## 6. Valores atipicos (outliers)

### Pregunta 12

Despues de la imputacion anterior, ¿cual es el valor **maximo** registrado en la columna `edad`?

In [34]:
# Tu codigo aqui
df["edad"].max()


np.float64(200.0)

### Pregunta 13

¿Cuantos clientes tienen una edad fuera de un rango logico, es decir, menor a 18 o mayor a 90 anios?

In [35]:
# Tu codigo aqui
((df["edad"] < 18) | (df["edad"] > 90)).sum()

np.int64(4)

### Pregunta 14

Usando el criterio del rango intercuartilico (IQR = Q3 - Q1; un valor es outlier si es menor a `Q1 - 1.5*IQR` o mayor a `Q3 + 1.5*IQR`), ¿cuantos valores atipicos se detectan en la columna `monto_invertido`?

In [36]:
# Tu codigo aqui
Q1 = df["monto_invertido"].quantile(0.25)
Q3 = df["monto_invertido"].quantile(0.75)

IQR = Q3 - Q1

lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR

((df["monto_invertido"] < lim_inf) |
 (df["monto_invertido"] > lim_sup)).sum()

np.int64(8)

## 7. Tabla cruzada: perfil de riesgo por rango de edad

### Pregunta 15

Elimina de `df` las filas con edad fuera del rango logico (menor a 18 o mayor a 90; mismo criterio de la Pregunta 13) y sobrescribe `df`. Crea una nueva columna `rango_edad` con:

```python
pd.cut(df["edad"], bins=[18, 30, 45, 60, 90],
       labels=["18-30", "31-45", "46-60", "61-90"], include_lowest=True)
```

Construye una tabla cruzada con `pd.crosstab(df["perfil_riesgo"], df["rango_edad"])`. Segun esa tabla, ¿cuantos clientes con `perfil_riesgo` **"Agresivo"** estan en el rango de edad **"18-30"**?

In [37]:
# Tu codigo aqui
# 1. Eliminar edades fuera del rango lógico
df = df[(df["edad"] >= 18) & (df["edad"] <= 90)].reset_index(drop=True)

# 2. Crear la columna rango_edad
df["rango_edad"] = pd.cut(
    df["edad"],
    bins=[18, 30, 45, 60, 90],
    labels=["18-30", "31-45", "46-60", "61-90"],
    include_lowest=True
)

# 3. Crear la tabla cruzada
tabla = pd.crosstab(df["perfil_riesgo"], df["rango_edad"])

# 4. Mostrar la tabla
tabla

rango_edad,18-30,31-45,46-60,61-90
perfil_riesgo,,,,
Agresivo,41,33,17,5
Conservador,15,27,20,35
Moderado,8,43,30,22
